In [ ]:
!pip install -q transformers peft bitsandbytes accelerate

In [ ]:
from peft import PeftModel
from transformers import AutoModelForCausalLM

base_model = AutoModelForCausalLM.from_pretrained(
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    torch_dtype="auto",
    device_map="auto"
)

model = PeftModel.from_pretrained(base_model, "/content/adapters")


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [ ]:
model = model.merge_and_unload()
print("MODEL MERGED SUCCESSFULLY")


MODEL MERGED SUCCESSFULLY


In [ ]:
model.save_pretrained("/content/quantized/model-fp16")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
)

tokenizer.save_pretrained("/content/quantized/model-fp16")

print("TOKENIZER SAVED")


TOKENIZER SAVED


In [ ]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_8bit=True
)

model_int8 = AutoModelForCausalLM.from_pretrained(
    "/content/quantized/model-fp16",
    quantization_config=bnb_config,
    device_map="auto"
)

model_int8.save_pretrained("/content/quantized/model-int8")

print("INT8 MODEL SAVED")


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

INT8 MODEL SAVED


In [ ]:
bnb_config_4 = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4"
)

model_int4 = AutoModelForCausalLM.from_pretrained(
    "/content/quantized/model-fp16",
    quantization_config=bnb_config_4,
    device_map="auto"
)

model_int4.save_pretrained("/content/quantized/model-int4")

print("INT4 MODEL SAVED")


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

INT4 MODEL SAVED


In [ ]:
!ls /content/quantized


model-fp16  model-int4	model-int8


In [ ]:
!git clone https://github.com/ggerganov/llama.cpp


Cloning into 'llama.cpp'...
remote: Enumerating objects: 80834, done.
remote: Counting objects: 100% (70/70), done.
remote: Compressing objects: 100% (53/53), done.
remote: Total 80834 (delta 40), reused 17 (delta 17), pack-reused 80764 (from 3)
Receiving objects: 100% (80834/80834), 300.28 MiB | 22.90 MiB/s, done.
Resolving deltas: 100% (58399/58399), done.


In [ ]:
!ls llama.cpp


AGENTS.md		       convert_lora_to_gguf.py	pocs
AUTHORS			       docs			poetry.lock
benches			       examples			pyproject.toml
build-xcframework.sh	       flake.lock		pyrightconfig.json
ci			       flake.nix		README.md
CLAUDE.md		       ggml			requirements
cmake			       gguf-py			requirements.txt
CMakeLists.txt		       grammars			scripts
CMakePresets.json	       include			SECURITY.md
CODEOWNERS		       LICENSE			src
common			       licenses			tests
CONTRIBUTING.md		       Makefile			tools
convert_hf_to_gguf.py	       media			vendor
convert_hf_to_gguf_update.py   models
convert_llama_ggml_to_gguf.py  mypy.ini


In [ ]:
!pip install -r /content/llama.cpp/requirements.txt


Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cpu, https://download.pytorch.org/whl/nightly, https://download.pytorch.org/whl/cpu, https://download.pytorch.org/whl/nightly, https://download.pytorch.org/whl/cpu, https://download.pytorch.org/whl/nightly
Ignoring torch: markers 'platform_machine == "s390x"' don't match your environment
Ignoring torch: markers 'platform_machine == "s390x"' don't match your environment


In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
)

tokenizer.save_pretrained("/content/quantized/model-fp16")

print("TOKENIZER SAVED")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


TOKENIZER SAVED


In [ ]:
!python /content/llama.cpp/convert_hf_to_gguf.py \
    /content/quantized/model-fp16 \
    --outfile /content/quantized/model.gguf \
    --outtype q8_0


INFO:hf-to-gguf:Loading model: model-fp16
INFO:hf-to-gguf:Model architecture: LlamaForCausalLM
INFO:hf-to-gguf:gguf: indexing model part 'model.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:output.weight,               torch.bfloat16 --> Q8_0, shape = {2048, 32000}
INFO:hf-to-gguf:token_embd.weight,           torch.bfloat16 --> Q8_0, shape = {2048, 32000}
INFO:hf-to-gguf:blk.0.attn_norm.weight,      torch.bfloat16 --> F32, shape = {2048}
INFO:hf-to-gguf:blk.0.ffn_down.weight,       torch.bfloat16 --> Q8_0, shape = {5632, 2048}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,       torch.bfloat16 --> Q8_0, shape = {2048, 5632}
INFO:hf-to-gguf:blk.0.ffn_up.weight,         torch.bfloat16 --> Q8_0, shape = {2048, 5632}
INFO:hf-to-gguf:blk.0.ffn_norm.weight,       torch.bfloat16 --> F32, shape = {2048}
INFO:hf-to-gguf:blk.0.attn_k.weight,         torch.bfloat16 --> Q8_0, shape = {2048, 256}
INFO:hf-to-gguf:blk.0.at

In [ ]:
!pip uninstall -y torchvision torchaudio



In [ ]:
!pip install -q transformers accelerate bitsandbytes sentencepiece

In [ ]:
import time
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_PATH = "/content/quantized/model-fp16"

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    device_map="auto",
    torch_dtype="auto"
)

prompt = "Explain artificial intelligence in simple words."

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

start = time.time()
output = model.generate(**inputs, max_new_tokens=120)
end = time.time()

tokens = output.shape[1] - inputs.input_ids.shape[1]
print("Tokens/sec:", tokens/(end-start))


`torch_dtype` is deprecated! Use `dtype` instead!


Tokens/sec: 0.3858186862712867


In [ ]:
!cp /content/quantized/model-fp16/tokenizer* /content/quantized/model-int8/
!cp /content/quantized/model-fp16/special_tokens_map.json /content/quantized/model-int8/

!cp /content/quantized/model-fp16/tokenizer* /content/quantized/model-int4/
!cp /content/quantized/model-fp16/special_tokens_map.json /content/quantized/model-int4/


In [ ]:
import time
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_PATH = "/content/quantized/model-int8"

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    device_map="auto",
    torch_dtype="auto"
)

prompt = "Explain artificial intelligence in simple words."

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

start = time.time()
output = model.generate(**inputs, max_new_tokens=120)
end = time.time()

tokens = output.shape[1] - inputs.input_ids.shape[1]
print("Tokens/sec:", tokens/(end-start))


/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


Tokens/sec: 0.07227723441092705


In [ ]:
import time
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_PATH = "/content/quantized/model-int4"

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    device_map="auto",
    torch_dtype="auto"
)

prompt = "Explain artificial intelligence in simple words."

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

start = time.time()
output = model.generate(**inputs, max_new_tokens=120)
end = time.time()

tokens = output.shape[1] - inputs.input_ids.shape[1]
print("Tokens/sec:", tokens/(end-start))


Tokens/sec: 0.023935651342848224


In [ ]:
!git clone https://github.com/ggml-org/llama.cpp


Cloning into 'llama.cpp'...
remote: Enumerating objects: 80834, done.
remote: Counting objects: 100% (73/73), done.
remote: Compressing objects: 100% (55/55), done.
remote: Total 80834 (delta 41), reused 18 (delta 18), pack-reused 80761 (from 3)
Receiving objects: 100% (80834/80834), 300.42 MiB | 2.40 MiB/s, done.
Resolving deltas: 100% (58424/58424), done.
Updating files: 100% (2277/2277), done.


In [ ]:
!cmake -B llama.cpp/build -S llama.cpp



-- The C compiler identification is GNU 11.4.0
-- The CXX compiler identification is GNU 11.4.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
CMAKE_BUILD_TYPE=Release
-- Found Git: /usr/bin/git (found version "2.34.1")
-- The ASM compiler identification is GNU
-- Found assembler: /usr/bin/cc
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD - Success
-- Found Threads: TRUE
-- Warning: ccache not found - consider installing it for faster compilation or disable this warning with GGML_CCACHE=OFF
-- CMAKE_SYSTEM_PROCESSOR: x86_64
-- GGML_SYSTEM_ARCH: x86
-- Including CPU backend
-- Found OpenMP_C: 

In [ ]:
!cmake --build llama.cpp/build --config Release -j4


[  0%] Built target build_info
[  1%] Built target xxhash
[  2%] Built target sha256
[  2%] Built target cpp-httplib
[  2%] Built target sha1
[  4%] Built target ggml-base
[  4%] Built target llama-llava-cli
[  5%] Built target llama-gemma3-cli
[  6%] Built target llama-minicpmv-cli
[  6%] Built target llama-qwen2vl-cli
[ 10%] Built target ggml-cpu
[ 11%] Built target ggml
[ 11%] Built target llama-gguf-hash
[ 12%] Built target llama-gguf
[ 44%] Built target llama
[ 44%] Built target test-c
[ 45%] Built target llama-simple
[ 45%] Built target llama-simple-chat
[ 52%] Built target common
[ 57%] Built target mtmd
[ 57%] Built target test-tokenizer-0
[ 58%] Built target test-grammar-parser
[ 58%] Built target test-sampling
[ 59%] Built target test-grammar-integration
[ 60%] Building CXX object tests/CMakeFiles/test-chat.dir/test-chat.cpp.o
[ 61%] Built target test-llama-grammar
[ 61%] Built target test-quantize-stats
[ 62%] Built target test-json-schema-to-grammar
[ 63%] Built target test

In [ ]:
!/content/llama.cpp/build/bin/llama-cli \
  -m /content/quantized/model.gguf \
  -p "Explain artificial intelligence simply" \
  -n 120



Loading model... |-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\| 


▄▄ ▄▄
██ ██
██ ██  ▀▀█▄ ███▄███▄  ▀▀█▄    ▄████ ████▄ ████▄
██ ██ ▄█▀██ ██ ██ ██ ▄█▀██    ██    ██ ██ ██ ██
██ ██ ▀█▄██ ██ ██ ██ ▀█▄██ ██ ▀████ ████▀ ████▀
                                    ██    ██
                                    ▀▀    ▀▀

build      : b8111-11c325c6e
model      : model.gguf
modalities : text

available commands:
  /exit or Ctrl+C     stop or exit
  /regen              regenerate the last response
  /clear              clear the chat history
  /read               add a text file


> Explain artificial intelligence simply

|-\|/-\|/-\| Artificial intelligence (AI) is a field of computer science that aims to create intelligent machines that can perform tasks like human beings. AI has been around for decades, and it has evolved significantly over time, with new technologies emerging and o

In [ ]:
!du -sh *


In [ ]:
!pip install -q bitsandbytes>=0.46.1 accelerate transformers sentencepiece


In [ ]:
import torch
import bitsandbytes as bnb

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))
print("bnb loaded")


In [ ]:
import time
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_PATH = "/content/quantized/model-fp16"

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    device_map="auto",
    torch_dtype="auto"
)

prompt = "Explain artificial intelligence in simple words."

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

start = time.time()
output = model.generate(**inputs, max_new_tokens=120)
end = time.time()

tokens = output.shape[1] - inputs.input_ids.shape[1]
print("Tokens/sec:", tokens/(end-start))


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

BASE_MODEL = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, device_map="auto")

messages = [
    {"role": "user", "content": "Explain artificial intelligence simply."}
]

inputs = tokenizer.apply_chat_template(
    messages,
    return_tensors="pt"
).to(model.device)

output = model.generate(inputs, max_new_tokens=120)

print(tokenizer.decode(output[0], skip_special_tokens=True))


<|user|>
Explain artificial intelligence simply. 
<|assistant|>
Artificial intelligence (AI) is a field of computer science that involves the development of computer systems that can perform tasks that would typically require human intelligence, such as reasoning, problem-solving, and decision-making. AI is based on the principles of computer science, machine learning, and computer vision, and it has the potential to revolutionize various industries, including healthcare, finance, transportation, and education.

AI can be divided into two main categories: machine learning and deep learning. Machine learning involves the use of algorithms to


In [ ]:
FINE_TUNED = "/content/quantized/model-fp16"

from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained(FINE_TUNED)
model = AutoModelForCausalLM.from_pretrained(FINE_TUNED, device_map="auto")

messages = [
    {"role": "user", "content": "Explain artificial intelligence simply."}
]

inputs = tokenizer.apply_chat_template(
    messages,
    return_tensors="pt"
).to(model.device)

output = model.generate(inputs, max_new_tokens=120)

print(tokenizer.decode(output[0], skip_special_tokens=True))


In [2]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [3]:
!ls

drive  sample_data


In [7]:
!ls /content/drive/MyDrive/Colab\ Notebooks


 Day3ipynb.ipynb  'GPU ON.ipynb'      Untitled0.ipynb   Untitled2.ipynb
 Day4.ipynb	   lora_train.ipynb   Untitled1.ipynb
